# 损失函数

上一章，我们构建了第一个神经网络模型，并用它预测了冰激凌的销量。但这个预测靠不靠谱，我们还不知道。

要知道预测是否准确，就离不开**损失函数**（Loss Function）。它的作用，就是衡量网络模型预测值与真实值之间的差距。

损失函数是模型训练的核心。它不仅告诉我们模型"错了多少"，更是指引模型自动改进的信号。

---

最直接的衡量差距的办法，是把**预测值**和**真实值**直接相减，它们之间的差距称为**误差**（Error）：

$$
\text{error} = y - p
$$

这里：
* $y$：真实结果，称为**标签值**（Label）；
* $p$：网络模型的预测结果，即**预测值**。

但是，直接使用误差有一个问题：当我们面对多个样本时，正的误差和负的误差会相互抵消，导致整体误差被低估。

在实践中，我们通常改用误差的平方，称为**平方误差**（Squared Error）：

$$
\text{squared error} = (y - p)^2
$$

使用平方误差有三个优点：
1. **消除正负号**：所有误差都变为正数，避免相互抵消；
2. **强调大误差**：误差越大，平方后惩罚越重，迫使模型优先修正大错误；
3. **处处可导**：这是使用梯度下降法自动优化模型参数的数学前提。

``💡 为什么"处处可导"如此关键？因为梯度下降法需要计算损失函数的导数来确定参数调整的方向。如果损失函数在某一点不可导（比如阶跃函数），梯度就无从计算，模型训练就会失败。``

## 均方误差（MSE）

当面对多个样本（例如 $n$ 天的销售数据）时，我们需要一个整体指标来衡量模型的性能。

常用的方法是计算所有样本平方误差的**平均值**，称为**均方误差**（Mean Squared Error，MSE）：

$$
\text{MSE} = \frac{1}{n} \sum\limits_{i=1}^{n} (y_i - p_i)^2
$$

这里：
* $y = [y_1, y_2, \ldots, y_n]$：$n$ 个真实结果；
* $p = [p_1, p_2, \ldots, p_n]$：$n$ 个预测结果。

MSE 越小，说明模型的整体预测越准确。MSE 为 0 意味着模型的预测与真实值完全一致，这在实践中几乎不可能，也不一定是好事（可能意味着过拟合）。

In [1]:
import numpy as np

## 数据

现在，让我们回到小明的冰激凌店。

当天晚上，销售数据出炉：一共卖出了 **165** 个冰激凌。

### 特征、标签

我们把真实结果称为**标签值**（Label），同样用 NumPy 数组来保存。

从这一章起，我们的每条数据都由两部分组成：
* **特征值**（Feature）：输入数据，即天气情况；
* **标签值**（Label）：对应的真实结果，即实际销量。

In [2]:
feature = np.array([28.1, 58.0])
label = np.array([165])

## 模型

我们继续使用上一章搭建的模型框架。

### 参数：权重、偏置

In [3]:
weight = np.ones((1, 2)) / 2
bias = np.zeros(1)

### 推理函数

In [4]:
def forward(x, w, b):
    return x @ w.T + b

### 损失函数（均方误差）

损失函数根据预测值 $p$ 和标签值 $y$ 计算出均方误差，结果称为**损失值**（Loss）。

损失值是一个**非负数**：
* 损失值越大，说明预测误差越大，模型越不准确；
* 损失值越小，说明预测越接近真实值，模型越准确。

In [5]:
def mse_loss(p, y):
    return np.mean(np.square(y - p))

## 验证

现在我们有了损失函数，可以正式评估模型的表现了。

### 推理

首先，用模型对今天的天气数据进行预测。

In [6]:
prediction = forward(feature, weight, bias)
print(f'prediction: {prediction}')

prediction: [43.05]


### 评估

有了损失函数，我们就可以量化模型预测的误差。

In [7]:
loss = mse_loss(prediction, label)
print(f'loss: {loss:.4f}')

loss: 14871.8025


损失值约为 `14872`。这个数字意味着什么？

我们使用的是**平方误差**，不像误差那样直观，但作为比较的基准非常有用：

* 损失值越小，表示模型越准确；
* 损失值的**变化趋势**才是关键，我们的目标是让它在训练过程中不断下降。

我们的第一个模型预测严重偏低。那么，如何让模型自动变得更准确？

## 课后练习

尝试修改权重和偏置的数值，看看能否让损失值降低？